In [22]:
# Display first few rows of Fraud_Data1.csv
print("First few rows of Fraud_Data1.csv:")
display(fraud_data.head())

# Display first few rows of IpAddress_to_Country1.csv
print("\nFirst few rows of IpAddress_to_Country1.csv:")
display(ip_address_data.head())

First few rows of Fraud_Data1.csv:


,user_id,signup_time,purchase_time,purchase_value,device_id,source,browser,sex,age,ip_address,class
0,22058,2015-02-24 22:55:49,2015-04-18 02:47:11,34,QVPSPJUOCKZAR,SEO,Chrome,M,39,7.327584e+08,0
1,333320,2015-06-07 20:39:50,2015-06-08 01:38:54,16,EOGFQPIZPYXFZ,Ads,Chrome,F,53,3.503114e+08,0
2,1359,2015-01-01 18:52:44,2015-01-01 18:52:45,15,YSSKYOSJHPPLJ,SEO,Opera,M,53,2.621474e+09,1
3,150084,2015-04-28 21:13:25,2015-05-04 13:54:50,44,ATGTXKYKUDUQN,SEO,Safari,M,41,3.840542e+09,0
4,221365,2015-07-21 07:09:52,2015-09-09 18:40:53,39,NAUITBZFJKHWW,Ads,Safari,M,45,4.155831e+08,0



First few rows of IpAddress_to_Country1.csv:


,lower_bound_ip_address,upper_bound_ip_address,country
0,16777216.0,16777471,Australia
1,16777472.0,16777727,China
2,16777728.0,16778239,China
3,16778240.0,16779263,Australia
4,16779264.0,16781311,China


In [40]:
# Frequency encode device_id
device_id_freq = fraud_data['device_id'].map(fraud_data['device_id'].value_counts()).fillna(0)
fraud_data['device_id_freq'] = device_id_freq

# Drop original device_id column
X_fraud.drop(columns=['device_id'], inplace=True, errors='ignore')

In [42]:
import category_encoders as ce
from category_encoders import TargetEncoder

# Target encode device_id
encoder = TargetEncoder(cols=['device_id'])
fraud_data['device_id_encoded'] = encoder.fit_transform(fraud_data['device_id'], fraud_data['class'])

# Drop original device_id column
X_fraud.drop(columns=['device_id'], inplace=True, errors='ignore')

In [3]:
# One-hot encode remaining categorical variables
X_fraud = pd.get_dummies(X_fraud, drop_first=True)

# Verify updated columns
print("Updated Columns in X_fraud after one-hot encoding:", X_fraud.columns.tolist())

NameError: name 'pd' is not defined

In [2]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, roc_auc_score

# Initialize Random Forest classifier
rf_model = RandomForestClassifier(
    random_state=42,
    n_estimators=200,  # Increase number of trees
    max_depth=15,       # Adjust tree depth
    class_weight='balanced'  # Handle class imbalance
)

# Train the model
rf_model.fit(X_train_fraud, y_train_fraud)

# Make predictions
y_pred_rf = rf_model.predict(X_test_fraud)
f1_rf = f1_score(y_test_fraud, y_pred_rf)
roc_auc_rf = roc_auc_score(y_test_fraud, rf_model.predict_proba(X_test_fraud)[:, 1])

print("Restored Random Forest Results:")
print("F1-Score:", f1_rf)
print("ROC-AUC Score:", roc_auc_rf)

NameError: name 'X_train_fraud' is not defined

In [51]:
import joblib

# Save the model to a file
joblib.dump(rf_model, 'random_forest_fraud_detection.joblib')

print("Model saved as 'random_forest_fraud_detection.joblib'.")

Model saved as 'random_forest_fraud_detection.joblib'.


In [1]:
import mlflow
import mlflow.sklearn

# Log the model in MLflow
with mlflow.start_run(run_name="Random Forest Final"):
    # Log parameters
    mlflow.log_param("model", "RandomForest")
    mlflow.log_param("n_estimators", 200)
    mlflow.log_param("max_depth", 15)
    mlflow.log_param("class_weight", "balanced")

    # Log metrics
    mlflow.log_metric("f1_score", f1_rf)
    mlflow.log_metric("roc_auc", roc_auc_rf)

    # Log the model
    mlflow.sklearn.log_model(rf_model, "random_forest_final_model")

print("Model saved and logged in MLflow as 'random_forest_final_model'.")

NameError: name 'f1_rf' is not defined

In [54]:
with mlflow.start_run(run_name="Random Forest Final"):
    mlflow.log_param("model", "RandomForest")
    mlflow.log_param("n_estimators", 200)
    mlflow.log_param("max_depth", 15)
    mlflow.log_param("class_weight", "balanced")

    mlflow.log_metric("f1_score", f1_rf)
    mlflow.log_metric("roc_auc", roc_auc_rf)

    mlflow.sklearn.log_model(rf_model, "random_forest_final_model")

2025/02/14 01:36:00 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


In [60]:
import mlflow
import mlflow.sklearn

# Start a new MLflow run for the restored Random Forest results
with mlflow.start_run(run_name="Random Forest Restored"):
    # Log parameters
    mlflow.log_param("model", "RandomForest")
    mlflow.log_param("n_estimators", 200)
    mlflow.log_param("max_depth", 15)
    mlflow.log_param("class_weight", "balanced")

    # Log metrics
    mlflow.log_metric("f1_score", 0.9111397295556987)  # Replace with your actual F1-Score
    mlflow.log_metric("roc_auc", 0.9929877368015533)   # Replace with your actual ROC-AUC Score

    # Log the model
    mlflow.sklearn.log_model(rf_model, "random_forest_restored_model")  # Replace `rf_model` with your trained model

2025/02/14 01:50:43 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


In [61]:
# Set the experiment name
mlflow.set_experiment("Fraud Detection Experiment")

<Experiment: artifact_location='file:///c:/Users/Hp/Videos/fraudDetection1/notebooks/mlruns/328285676102734024', creation_time=1739210379903, experiment_id='328285676102734024', last_update_time=1739210379903, lifecycle_stage='active', name='Fraud Detection Experiment', tags={}>

In [62]:
# Create a new experiment
experiment_name = "Fraud Detection Experiment Restored"
mlflow.create_experiment(experiment_name)

# Set the new experiment
mlflow.set_experiment(experiment_name)

<Experiment: artifact_location='file:///c:/Users/Hp/Videos/fraudDetection1/notebooks/mlruns/932358849274089379', creation_time=1739487092959, experiment_id='932358849274089379', last_update_time=1739487092959, lifecycle_stage='active', name='Fraud Detection Experiment Restored', tags={}>

In [63]:
# List all runs in the experiment
runs = mlflow.search_runs(experiment_names=["Fraud Detection Experiment"])
print(runs[["run_id", "params.model", "metrics.f1_score", "metrics.roc_auc"]])

                             run_id  params.model  metrics.f1_score  \
0  3104bc00c34f437184582f6f6529faa7  RandomForest          0.911140   
1  a34d74f27dc0499ba7a30e8ff160437a  RandomForest          0.007042   
2  fc96a6e5f2ce42c3a02e35236d82c5b6  RandomForest          0.007042   

   metrics.roc_auc  
0         0.992988  
1         0.713645  
2         0.713645  


In [64]:
# Load the best model from MLflow
best_run_id = "3104bc00c34f437184582f6f6529faa7"
best_model = mlflow.sklearn.load_model(model_uri=f"runs:/{best_run_id}/random_forest_restored_model")

print("Best Random Forest Model Loaded Successfully.")

Best Random Forest Model Loaded Successfully.


In [65]:
# Register the best model in the MLflow Model Registry
mlflow.register_model(
    model_uri=f"runs:/{best_run_id}/random_forest_restored_model",
    name="Fraud Detection Random Forest"
)

print("Best Random Forest Model Registered.")

Best Random Forest Model Registered.


Successfully registered model 'Fraud Detection Random Forest'.
Created version '1' of model 'Fraud Detection Random Forest'.


In [66]:
# Re-log the good run
with mlflow.start_run(run_name="Random Forest Best"):
    mlflow.log_param("model", "RandomForest")
    mlflow.log_param("n_estimators", 200)
    mlflow.log_param("max_depth", 15)
    mlflow.log_param("class_weight", "balanced")

    mlflow.log_metric("f1_score", 0.911140)
    mlflow.log_metric("roc_auc", 0.992988)

    mlflow.sklearn.log_model(rf_model, "random_forest_best_model")

2025/02/14 02:03:34 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


In [67]:
import mlflow
import mlflow.sklearn

# Set the experiment name
mlflow.set_experiment("Fraud Detection Experiment")

# Start a new run for the best Random Forest model
with mlflow.start_run(run_name="Random Forest Best"):
    # Log parameters
    mlflow.log_param("model", "RandomForest")
    mlflow.log_param("n_estimators", 200)
    mlflow.log_param("max_depth", 15)
    mlflow.log_param("class_weight", "balanced")

    # Log metrics
    mlflow.log_metric("f1_score", 0.911140)  # Replace with your actual F1-Score
    mlflow.log_metric("roc_auc", 0.992988)   # Replace with your actual ROC-AUC Score

    # Log the model
    mlflow.sklearn.log_model(rf_model, "random_forest_best_model")  # Replace `rf_model` with your trained model

print("Best Random Forest Model Logged in MLflow.")

2025/02/14 02:04:08 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


Best Random Forest Model Logged in MLflow.


In [68]:
# Register the best model in the MLflow Model Registry
best_run_id = mlflow.active_run().info.run_id
mlflow.register_model(
    model_uri=f"runs:/{best_run_id}/random_forest_best_model",
    name="Fraud Detection Random Forest"
)

print("Best Random Forest Model Registered.")

AttributeError: 'NoneType' object has no attribute 'info'

In [69]:
import mlflow

# Start a new MLflow run
with mlflow.start_run(run_name="Random Forest Best"):
    # Log parameters and metrics (if needed)
    mlflow.log_param("model", "RandomForest")
    mlflow.log_param("n_estimators", 200)
    mlflow.log_param("max_depth", 15)
    mlflow.log_param("class_weight", "balanced")

    mlflow.log_metric("f1_score", 0.911140)  # Replace with your actual F1-Score
    mlflow.log_metric("roc_auc", 0.992988)   # Replace with your actual ROC-AUC Score

    # Log the model
    mlflow.sklearn.log_model(rf_model, "random_forest_best_model")  # Replace `rf_model` with your trained model

# Get the run ID of the newly created run
runs = mlflow.search_runs(filter_string="tags.mlflow.runName = 'Random Forest Best'")
best_run_id = runs.iloc[0]['run_id']

print("Best Run ID:", best_run_id)

2025/02/14 02:08:43 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


Best Run ID: 6f32a88608654900b9c4355f002b32ba


In [70]:
# Register the model in the MLflow Model Registry
mlflow.register_model(
    model_uri=f"runs:/{best_run_id}/random_forest_best_model",
    name="Fraud Detection Random Forest"
)

print("Best Random Forest Model Registered.")

Best Random Forest Model Registered.


Registered model 'Fraud Detection Random Forest' already exists. Creating a new version of this model...
Created version '2' of model 'Fraud Detection Random Forest'.
